# TP Jour 4 — Tests d'Hypothèses Statistiques, Segmentation & Restitution BI

### Objectifs de la séance
1. Valider scientifiquement deux hypothèses métier sur votre dataset à l'aide de **tests statistiques adaptés** (`scipy.stats`).
2. Construire une **segmentation / typologie non supervisée (K-Means)** combinant agrégation distribuée **PySpark**, transformation d'asymétrie (`log1p`), standardisation (`StandardScaler`) et labellisation métier vers **`/lakehouse/gold/`**.
3. Développer votre **Dashboard Streamlit interactif** dans `app/dashboard.py` (`http://localhost:8501`) en documentant chaque requête SQL et chaque choix statistique.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("J4_Stats_Segmentation_Dashboard")
    .getOrCreate()
)

:: loading settings :: url = jar:file:/opt/venv/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d7d2618b-dc5e-4d33-8f40-a1c19b296d11;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.1 in central
	found

## 1. Inférence Statistique : Tester vos Hypothèses Métier (`scipy.stats`)

Choisissez **au moins deux tests statistiques** pertinents pour votre sujet parmi :
- **Comparaison de 2 groupes indépendants** sur une variable asymétrique : **Mann-Whitney U** (`stats.mannwhitneyu`).
- **Comparaison de $k \ge 3$ groupes** : **Kruskal-Wallis** (`stats.kruskal`).
- **Lien monotone entre 2 variables quantitatives** non gaussiennes : **Corrélation de rangs de Spearman** (`stats.spearmanr`).
- **Dépendance entre 2 variables qualitatives** : **Test du $\chi^2$** (`stats.chi2_contingency`).

### Exercice 1.1 — Formuler $H_0 / H_1$, projeter les colonnes utiles depuis Silver et conclure
> **Bonne pratique Big Data :** Ne faites jamais `.toPandas()` sur toute une table Silver brute ! Filtrez et sélectionnez (`select(...)`) uniquement les colonnes nécessaires côté PySpark avant de rapatrier les données vers le driver.

**Hypothèses sur les vols non annulés dont le retard est connu :**
- **Mann–Whitney bilatéral** : $H_0$ = mêmes distributions de retard le week-end et en semaine ; $H_1$ = distributions différentes. Ce test de rangs convient à une distribution asymétrique.
- **Spearman bilatéral** : $H_0$ = absence de corrélation monotone entre distance et retard à l'arrivée ; $H_1$ = corrélation monotone non nulle.

Seuil de 5 %. Les résultats concernent un échantillon aléatoire plafonné ; ils sont exploratoires. Les vols peuvent être dépendants (même ligne, jour ou compagnie) : les p-valeurs supposant l'indépendance doivent être interprétées avec prudence. Corrélation et différence ne prouvent pas une causalité.

In [2]:
SILVER_MAIN_PATH = "/lakehouse/silver/flights_enriched"
silver = spark.read.format("delta").load(SILVER_MAIN_PATH)
df_sample = (silver.filter((F.col("cancelled") == 0) & F.col("arrival_delay").isNotNull() & F.col("distance").isNotNull() & F.col("is_weekend").isNotNull())
    .select("arrival_delay", "distance", "is_weekend")
    .sample(fraction=0.001, seed=42).limit(10000).toPandas())
weekend = df_sample.loc[df_sample["is_weekend"], "arrival_delay"].to_numpy()
weekday = df_sample.loc[~df_sample["is_weekend"], "arrival_delay"].to_numpy()
if len(weekend) < 2 or len(weekday) < 2 or df_sample["distance"].nunique() < 2 or df_sample["arrival_delay"].nunique() < 2:
    raise ValueError("Échantillon insuffisant ou constant pour les tests statistiques.")
u, p_u = stats.mannwhitneyu(weekend, weekday, alternative="two-sided", method="asymptotic")
rank_biserial = 2 * u / (len(weekend) * len(weekday)) - 1
rho, p_rho = stats.spearmanr(df_sample["distance"], df_sample["arrival_delay"])
if not np.isfinite([u, p_u, rho, p_rho]).all():
    raise ValueError("Les tests ont produit un résultat non fini.")
print(f"Échantillon : {len(df_sample):,} vols ; week-end : {len(weekend):,} ; semaine : {len(weekday):,}.")
print(f"Mann–Whitney : U = {u:.0f}, p = {p_u:.3g}, effet rank-bisérial = {rank_biserial:.3f} (week-end moins semaine).")
print("H0 distributions identiques :", "rejetée à 5 %" if p_u < 0.05 else "non rejetée à 5 %")
print(f"Spearman : rho = {rho:.3f}, p = {p_rho:.3g}.")
print("H0 absence de corrélation :", "rejetée à 5 %" if p_rho < 0.05 else "non rejetée à 5 %")
print("Ces tests ne démontrent pas une relation causale ; tenir compte de la dépendance possible entre vols.")

/opt/venv/lib/python3.15/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/18 15:06:27 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Échantillon : 5,762 vols ; week-end : 1,520 ; semaine : 4,242.
Mann–Whitney : U = 3165591, p = 0.294, effet rank-bisérial = -0.018 (week-end moins semaine).
H0 distributions identiques : non rejetée à 5 %
Spearman : rho = -0.093, p = 1.38e-12.
H0 absence de corrélation : rejetée à 5 %
Ces tests ne démontrent pas une relation causale ; tenir compte de la dépendance possible entre vols.


## 2. Segmentation / Typologie par Clustering K-Means (`/lakehouse/gold/`)

Quelle que soit la thématique de votre dataset Kaggle, identifiez l'**entité métier pertinente à segmenter** (ex. clients/utilisateurs, stations, zones géographiques, produits, lignes de transport, capteurs...) :
1. **Agrégation PySpark** : résumez chaque entité par **3 à 5 features numériques comportementales** (fréquence, volume, valeur médiane/totale, récence, taux d'incident...).
2. **Prétraitement statistique** : appliquez `np.log1p(...)` sur les features positives asymétriques puis centrez-réduisez avec `StandardScaler()` afin qu'aucune variable à grande échelle n'écrase la distance euclidienne du K-Means.
3. **Interprétation métier** : inspectez les médianes par cluster (`groupby("cluster_id").median()`), attribuez un **nom explicite** à chaque segment (`colonne "segment"`), puis enregistrez la table dans **`/lakehouse/gold/<nom_table_segments>`**.

Ici l'entité est l'aéroport de départ : seuls les aéroports comptant au moins 100 vols et des indicateurs complets sont segmentés. Le taux de retard est calculé sur les vols non annulés avec retard connu ; le taux d'annulation porte sur les statuts connus. Les étiquettes faible/intermédiaire/élevé sont **relatives aux médianes des trois clusters**, pas des seuils absolus.

In [3]:
GOLD_SEGMENTS_PATH = "/lakehouse/gold/flights_airport_segments"
known_arrival = (F.col("cancelled") == 0) & F.col("arrival_delay").isNotNull()
features = ["n_vols", "distance_moyenne", "retard_positif_moyen", "taux_retard", "taux_annulation"]
airports = (silver.groupBy("origin_airport")
    .agg(F.max("origin_airport_name").alias("airport_name"), F.count("*").alias("n_vols"),
         F.avg("distance").alias("distance_moyenne"),
         F.avg(F.when(known_arrival, F.greatest(F.col("arrival_delay"), F.lit(0)))).alias("retard_positif_moyen"),
         (100 * F.avg(F.when(known_arrival, (F.col("arrival_delay") >= 15).cast("double")))).alias("taux_retard"),
         (100 * F.avg(F.col("cancelled").cast("double"))).alias("taux_annulation"))
    .filter(F.col("n_vols") >= 100)
    .withColumn("airport_name", F.coalesce(F.col("airport_name"), F.concat(F.lit("Code non référencé : "), F.col("origin_airport")))))
before_drop = airports.count()
airports = airports.dropna(subset=features)
eligible = airports.count()
print(f"Aéroports ≥ 100 vols : {before_drop} ; exclus pour indicateurs manquants : {before_drop - eligible}.")
if eligible < 3 or eligible > 10000:
    raise ValueError(f"Nombre d'aéroports éligibles hors limites pour K-Means : {eligible}.")
entities_pd = airports.orderBy("origin_airport").toPandas()
X = entities_pd[features].astype(float).copy()
if not np.isfinite(X.to_numpy()).all() or (X[["n_vols", "distance_moyenne", "retard_positif_moyen"]] < 0).any().any():
    raise ValueError("Indicateurs non finis ou négatifs incompatibles avec log1p.")
X[["n_vols", "distance_moyenne", "retard_positif_moyen"]] = np.log1p(X[["n_vols", "distance_moyenne", "retard_positif_moyen"]])
scaled = StandardScaler().fit_transform(X)
if np.unique(scaled, axis=0).shape[0] < 3:
    raise ValueError("Trois profils distincts au minimum sont nécessaires pour K-Means.")
entities_pd["cluster_id"] = KMeans(n_clusters=3, random_state=42, n_init=20).fit_predict(scaled)
cluster_medians = entities_pd.groupby("cluster_id")[features].median()
ordered_clusters = cluster_medians.sort_values("taux_retard").index.tolist()
labels = ["Retard relatif faible", "Retard relatif intermédiaire", "Retard relatif élevé"]
entities_pd["segment"] = entities_pd["cluster_id"].map(dict(zip(ordered_clusters, labels)))
print("Médianes des indicateurs par cluster :")
display(cluster_medians.round(2))
print("Répartition des segments :")
display(entities_pd.groupby("segment").size().rename("aeroports"))
rows = [(r.origin_airport, r.airport_name, int(r.n_vols), float(r.distance_moyenne), float(r.retard_positif_moyen), float(r.taux_retard), float(r.taux_annulation), int(r.cluster_id), r.segment) for r in entities_pd.itertuples(index=False)]
schema = "origin_airport string, airport_name string, n_vols long, distance_moyenne double, retard_positif_moyen double, taux_retard double, taux_annulation double, cluster_id int, segment string"
(
    spark.createDataFrame(rows, schema)
    .coalesce(2)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_SEGMENTS_PATH)
)
print(f"Table de segmentation enregistrée dans {GOLD_SEGMENTS_PATH} ({len(entities_pd):,} entités) !")

Aéroports ≥ 100 vols : 534 ; exclus pour indicateurs manquants : 0.


/opt/venv/lib/python3.15/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Médianes des indicateurs par cluster :


,n_vols,distance_moyenne,retard_positif_moyen,taux_retard,taux_annulation
cluster_id,,,,,
0,577.0,496.10,7.70,10.81,0.45
1,10422.0,716.32,12.08,17.27,1.45
2,967.5,299.35,13.68,18.10,3.02


Répartition des segments :


segment
Retard relatif faible           219
Retard relatif intermédiaire    163
Retard relatif élevé            152
Name: aeroports, dtype: int64

Table de segmentation enregistrée dans /lakehouse/gold/flights_airport_segments (534 entités) !


## 3. Livrable Final : Votre Application `app/dashboard.py` sur Streamlit (`:8501`)

Complétez maintenant **`app/dashboard.py`** pour valoriser vos tables `/lakehouse/gold/` :
- **Architecture respectée** : l'application Streamlit ne doit lire **que** des tables pré-calculées de la couche **Gold** (`/lakehouse/gold/...`), jamais les fichiers CSV bruts !
- **Interactivité** : au moins un filtre utilisateur (`st.selectbox`, `st.multiselect`, `st.slider`) et 3 à 4 visualisations Plotly Express complémentaires.
- **Rigueur explicative** : sous chaque visualisation clé, intégrez un `st.expander("Requête SQL & Justification statistique")` affichant la requête Spark SQL ayant produit la table Gold et justifiant le choix des indicateurs statistiques retenus.

In [4]:
# Vérification finale de l'inventaire complet de votre couche Gold sur HDFS
jvm = spark._jvm
fs = jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
PathHDFS = jvm.org.apache.hadoop.fs.Path
gold_root = PathHDFS("/lakehouse/gold")

gold_tables = [
    s.getPath().getName()
    for s in fs.listStatus(gold_root)
    if s.isDirectory() and fs.exists(PathHDFS(s.getPath(), "_delta_log"))
]
assert len(gold_tables) > 0, "Aucune table Delta trouvée dans /lakehouse/gold/ !"

print("=== Tables Delta disponibles dans /lakehouse/gold/ pour Streamlit ===")
for t in sorted(gold_tables):
    cnt = spark.read.format("delta").load(f"/lakehouse/gold/{t}").count()
    print(f" - /lakehouse/gold/{t:<25} : {cnt:>9,} lignes")

=== Tables Delta disponibles dans /lakehouse/gold/ pour Streamlit ===


 - /lakehouse/gold/flights_airline_pareto    :        14 lignes


 - /lakehouse/gold/flights_airline_top_origin :        14 lignes
 - /lakehouse/gold/flights_airport_segments  :       534 lignes
 - /lakehouse/gold/flights_monthly           :        12 lignes
